# Eval-Driven Development for Agents (Offline Loop)
## From a Failing Trace to a Passing CI

**Eval-driven development** is the discipline of treating agent quality as a measurable, testable property of your system rather than something you hope works. It has four moving parts:

1. **A dataset** — input → expected-output pairs that capture what "correct" looks like for your agent
2. **Evaluators** — functions that score agent runs against those expectations
3. **Experiments** — runs of the dataset through different agent versions, scored and compared
4. **A CI gate** — automated assertion that scores stay above threshold before code merges

Per LangChain's [Agent Evaluation Readiness Checklist](https://www.langchain.com/blog/agent-evaluation-readiness-checklist), agent eval has two complementary halves:

- **Regression evals** — verify the agent still passes known cases (~100% pass rate; this is the CI gate)
- **Capability evals** — track pass rate on harder examples climbing over time (~30–60% baseline; tracking, not gating)

This notebook walks through the offline portion of that loop live, using a customer support triage agent as the demonstration vehicle. The four-part pattern transfers to any agent with a written policy — sales agents, ops copilots, internal assistants.

**Three vocabulary notes from the readiness checklist** that show up below:

- **Trace-level (full-turn)** — score what the agent produced for one user message, including the trajectory it took. The checklist explicitly says start here.
- **Run-level** — individual tool calls inside a trace. Useful for debugging specific steps.
- **Thread-level** — multi-turn conversations. Matters when context grows over many turns; not in scope for this single-turn demo.

> **Reference docs cited throughout this notebook:**
> [Agent Evaluation Readiness Checklist](https://www.langchain.com/blog/agent-evaluation-readiness-checklist) ·
> [LangSmith Evaluation Quickstart](https://docs.langchain.com/langsmith/evaluation-quickstart) ·
> [LangSmith pytest integration](https://docs.langchain.com/langsmith/pytest) ·
> [HumanInTheLoopMiddleware](https://docs.langchain.com/oss/python/langchain/human-in-the-loop) ·
> [openevals](https://github.com/langchain-ai/openevals) ·
> [agentevals](https://github.com/langchain-ai/agentevals)

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
# Notebook runs land in the same project as the script-path runs (set in .env
# as agent-eval-workshop). The CI workflow uses agent-eval-workshop-ci so PR
# runs stay isolated from demo-rehearsal runs.
from src.agent import agent
from src.seed_data import SEED_EXAMPLES, CAPABILITY_EXAMPLES
print("Agent ready.")

## First — the happy path

Before exercising the trap, let's see the agent succeed on a clean case. We're picking **`ex-001`** from `SEED_EXAMPLES`: a free-tier customer (`cust_002`) asking how to reset their password.

**Why start here:** the readiness checklist's first dataset-construction principle is *"ensure every task is unambiguous, with a reference solution that proves it's solvable."* Happy paths anchor the dataset — they verify the agent CAN succeed before we test where it fails. Without a green baseline, regression scores have nothing to regress against.

**Expected agent trajectory:**

1. Call `get_customer_plan` → confirm `cust_002` is on free tier
2. Call `search_kb` → pull `kb-002` (password reset policy)
3. Compose response citing `kb-002`

In LangSmith, this trace becomes our reference for "what right looks like" — three tool calls, one citation, no escalation, no refund.

In [ ]:
happy = next(e for e in SEED_EXAMPLES if e["id"] == "ex-001")
config = {"configurable": {"thread_id": "demo-happy"}}
result = agent.invoke(
    {"messages": [{"role": "user", "content": happy["inputs"]["message"]}]},
    config=config,
    version="v2",
)
print(result.value["messages"][-1].content)

# >>> NARRATION (~3 min): Run cell. Output cites kb-002. Then SWITCH TO
#     LANGSMITH and walk through the trace step-by-step:
#     "Look at the trace. The agent first called get_customer_plan to fetch
#      cust_002's account — free tier. Then it called search_kb with the
#      query 'password reset' — pulled kb-002. Then it composed the response
#      citing kb-002. Three tool calls, one citation, full traceability.
#      This is what tracing gives you for free with create_agent — every
#      reasoning step, every tool call, every LLM message, end-to-end."
# >>> Then switch back to notebook for cell 5 markdown.

## Now the trap — `ex-012`

`ex-012` is a **deliberate edge case**: a free-tier customer (`cust_002`) asking for a refund. The agent's policy is encoded in the system prompt:

> *"Never refund a free-tier customer."*

Per the readiness checklist's grader-design section: *"Test both positive cases (behavior should occur) and negative cases (behavior should not occur)."* `ex-012` is the canonical negative case — the agent must NOT call `issue_refund` here. It's the trap that drives the regression demo: deleting that one prompt line causes the agent to start refunding free-tier customers, and `refund_safety` catches the regression instantly.

**The HITL safety net.** Even if the agent's policy reasoning fails, `HumanInTheLoopMiddleware` intercepts the `issue_refund` tool call before execution. Watch for `Interrupted? True` in the output below — that's the middleware firing. In production, a human reviewer would approve or reject before the refund actually goes through; in this notebook the eval target auto-approves so we measure agent *intent* (would the agent have filed a refund it shouldn't have?).

**Two-layer protection:**

- **Eval discipline (upstream)** — `refund_safety` catches the regression in CI, before merging
- **HITL middleware (runtime)** — catches anything that slipped past CI

Eval-driven development is what makes the runtime gate fire rarely instead of constantly.

In [ ]:
from langgraph.types import Command
trap = next(e for e in SEED_EXAMPLES if e["id"] == "ex-012")
config = {"configurable": {"thread_id": "demo-trap"}}
result = agent.invoke(
    {"messages": [{"role": "user", "content": trap["inputs"]["message"]}]},
    config=config,
    version="v2",
)
print("Interrupted?", bool(result.interrupts))
print(result.value["messages"][-1].content)

# >>> NARRATION (~3 min):
# >>> Step 1: Run cell. Result shows Interrupted? True.
# >>> Step 2: SWITCH TO LANGSMITH trace. Point at the issue_refund call:
#     "See here — the agent called issue_refund with $50 to cust_002. The
#      middleware caught it. In production this would route to a human
#      reviewer. But notice: the agent WANTED to refund a free-tier customer,
#      which violates policy. HITL caught it — but eval-driven development
#      is what catches it BEFORE production."
# >>> Step 2b: SIDE-TAB FLASH src/agent.py (~10 sec):
#     "Quick aside — the agent itself is one create_agent() call with
#      HumanInTheLoopMiddleware configured to interrupt on issue_refund.
#      That's the entire wiring. Policy lives in the system prompt, gating
#      lives in middleware config — the framework gives you both."
# >>> Step 2c: SIDE-TAB FLASH evals/target.py (~15 sec):
#     "And the eval target auto-approves the interrupt to measure agent
#      intent — Command(resume={'decisions': [{'type': 'approve'}]}). Five
#      lines. version='v2' uses the LangGraph 1.1+ result surface so we
#      read result.value and result.interrupts directly. The same target
#      handles single-turn and multi-turn examples — that's how ex-021
#      works without a separate runner."
# >>> Step 3: Deliver the VERBATIM 5-sentence HITL narration.
# >>>         Source of truth: tests/notebook_narration.py
# >>>         Verified post-build by tests/test_notebook_integrity.py
# >>> ===== VERBATIM-START — copy character-for-character =====
#     "issue_refund SUBMITS A REQUEST — it doesn't execute the refund.
#      The customer is told 'submitted, you'll get an email.'
#      HITL middleware then gates the internal review step before the request hits the processing queue.
#      The eval auto-approves to measure agent intent; production has a real reviewer.
#      Eval-driven discipline upstream means this gate fires rarely instead of constantly."
# >>> ===== VERBATIM-END =====

## Catching it BEFORE production

The trap above showed the agent attempting an unsafe action. HITL caught it at runtime — but a runtime catch is the LAST line of defense. The discipline of **eval-driven development** is to catch failures in development, before code merges.

**The four-part eval reliability loop in action below:**

1. **Dataset** — `support-triage-v1` (21 regression examples + 1 multi-turn demo, `ex-021`) already upserted to LangSmith
2. **Evaluators** — five functions across the three canonical types:
   - Heuristic (deterministic): `classification_correct`, `refund_safety`, `escalation_correctness` — `evals/heuristic_evaluators.py`
   - LLM-as-judge: `kb_grounding_judge` (uses `o3-mini` via [`openevals.create_llm_as_judge`](https://github.com/langchain-ai/openevals)) — `evals/llm_judge_evaluators.py`
   - Trajectory: `trajectory_superset` (uses `agentevals`) — `evals/trajectory_evaluators.py`
3. **Experiments** — three named runs we'll compare in LangSmith side-by-side: `v1-baseline` (clean prompt), `v2-removed-guardrail` (regression introduced), `v3-restored` (regression fixed)
4. **CI gate** — pytest runs in GitHub Actions on every PR. **Both regression and capability suites run together; gating is via assertions inside the regression tests, not via a marker filter.** Regression hard-asserts on `refund_safety == 1` (a regression fails the PR). Capability has no asserts — it logs scores to LangSmith for tracking but never blocks merging. The full workflow:

```yaml
# .github/workflows/evals.yml — runs on every pull request
name: Agent Quality Evals
on:
  pull_request:
    branches: [main]
  push:
    branches: [main]
jobs:
  evals:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: astral-sh/setup-uv@v3
      - run: uv python install 3.11
      - run: uv sync --frozen
      - env:
          OPENAI_API_KEY: ${{ secrets.OPENAI_API_KEY }}
          LANGSMITH_API_KEY: ${{ secrets.LANGSMITH_API_KEY }}
          LANGSMITH_TRACING: "true"
          LANGSMITH_PROJECT: "agent-eval-workshop-ci"
        run: uv run pytest tests/ --langsmith-output -v
```

The next code cell runs the regression evaluators against the dataset (the wow moment — a regression caught visually). The cell after that runs pytest exactly as CI does, demonstrating the gate that protects every merge to main.

> *Per the [Agent Evaluation Readiness Checklist](https://www.langchain.com/blog/agent-evaluation-readiness-checklist):* *"Integrate regression evals into your CI/CD pipeline with automated quality gates."* That's exactly what's wired up here.

## What each evaluator actually does

Before running the experiment, let's look at the source of one evaluator from each of the three canonical types. All three follow the same `(inputs, outputs, reference_outputs) -> {"key", "score", "comment"}` signature from the [LangSmith Evaluation Quickstart](https://docs.langchain.com/langsmith/evaluation-quickstart) — runner-agnostic, so the same evaluators work in `client.evaluate(...)` here AND in pytest under CI.

### Heuristic — `refund_safety` (deterministic Python, ~10 lines)

Source: `evals/heuristic_evaluators.py`

```python
def refund_safety(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    """Refund safety: should_refund must match the actual refund call."""
    tools_called = outputs.get("tool_calls", [])
    refunded = "issue_refund" in tools_called
    should_refund = reference_outputs.get("should_refund", False)

    if refunded and not should_refund:
        return {"key": "refund_safety", "score": 0,
                "comment": "Agent issued refund when it should not have"}
    if not refunded and should_refund:
        return {"key": "refund_safety", "score": 0,
                "comment": "Agent failed to issue refund when it should have"}
    return {"key": "refund_safety", "score": 1}
```

A heuristic just checks tool-call presence against the reference. Deterministic, fast, free — the right shape for **structural / mechanical properties**. This is the load-bearing CI gate (`pytest -m "not capability"` hard-asserts on `refund_safety == 1`).

### LLM-as-judge — `kb_grounding_judge` (uses `o3-mini` + `openevals`)

Source: `evals/llm_judge_evaluators.py`

```python
def kb_grounding_judge(inputs: dict, outputs: dict, reference_outputs: dict):
    """LLM-as-judge for KB grounding using o3-mini."""
    if not reference_outputs.get("expected_kb_doc_id"):
        return {"key": "kb_grounding", "score": None,
                "comment": "skipped: no KB citation expected for this example"}

    evaluator = create_llm_as_judge(
        prompt=KB_GROUNDING_PROMPT,
        model="openai:o3-mini",
        feedback_key="kb_grounding",
    )
    try:
        return evaluator(inputs=inputs, outputs=outputs,
                         reference_outputs=reference_outputs)
    except LengthFinishReasonError as e:
        return {"key": "kb_grounding", "score": None,
                "comment": f"skipped: o3-mini exhausted output token limit ({e})"}
```

LLM-judge wraps a rubric prompt + reasoning model (`o3-mini`) via `openevals.create_llm_as_judge`. **Three load-bearing patterns:**

1. **Early skip** — examples that don't expect a citation return `score=None`, not 0. Otherwise the headline average tanks because half the dataset has no citation expectation.
2. **`try/except LengthFinishReasonError`** — caught a real `o3-mini` runaway during rehearsal (99,998 completion tokens before the API capped). The guard returns `score=None` instead of hanging the eval for ~10 minutes.
3. **No CI hard-assert on judge scores** — judge scores get logged for visibility but not gated on. Until the judge is calibrated via Align Eval (see calibration cell below), gating on it would fail PRs on judge variance rather than real regressions.

The right shape for **subjective quality properties** (faithfulness, tone, citation grounding). Requires human calibration — that's the next cell beat.

### Trajectory — `trajectory_superset` (heuristic over the tool sequence)

Source: `evals/trajectory_evaluators.py`

```python
def trajectory_superset(inputs: dict, outputs: dict, reference_outputs: dict):
    """Heuristic: every expected tool must appear in the actual tool sequence."""
    expected_tools = reference_outputs.get("expected_tools", [])
    if not expected_tools:
        return {"key": "trajectory_superset", "score": None, "comment": "no reference"}

    actual = outputs.get("tool_calls", [])
    score = 1 if all(t in actual for t in expected_tools) else 0
    return {"key": "trajectory_superset", "score": score,
            "comment": f"expected={expected_tools} actual={actual}"}
```

Trajectory evaluators score the **path**, not just the answer — catches cases where the agent reaches a correct-looking answer through a wrong path. [`agentevals`](https://github.com/langchain-ai/agentevals) also ships an LLM-judge variant (`create_trajectory_llm_as_judge`) for cases where strict superset matching is too rigid.

---

**The takeaway: same canonical signature, different machinery.** Heuristics are 10 lines of Python; LLM-judges wrap [`openevals`](https://github.com/langchain-ai/openevals); trajectory checks live in `agentevals`. The runner (`client.evaluate` next cell) doesn't know which type it's executing — it just calls each evaluator with the same three positional dicts.

> *LIVE NARRATION (~75 sec):*
>
> *"Quick look at how these are built before we run them. `refund_safety` is a deterministic check — IF customer plan is 'free' AND `issue_refund` was called, score 0; else 1. That's a heuristic — fast, cheap, deterministic, gates CI. `kb_grounding_judge` is an LLM-judge — `o3-mini` reads the response and scores whether it cited the right KB doc. Subjective, costs money per run, gets recalibrated over time — tracks but doesn't gate. `trajectory_superset` is a path check — did the agent's tool sequence include the canonical sequence? That's a trajectory evaluator. Same target, three different evaluator types, three different failure modes caught."*
>
> *Then SIDE-TAB FLASH the full files (~5 sec each, no deep dive):*
>
> *— `evals/heuristic_evaluators.py`: "Three heuristics in this file — `classification_correct`, `refund_safety`, `escalation_correctness`. Same signature, ~10 lines each."*
>
> *— `evals/llm_judge_evaluators.py`: "One LLM-judge in this file. The rubric prompt is what makes it work — `KB_GROUNDING_PROMPT` defines the 0/0.5/1 scoring criteria. Calibration via Align Eval iterates on this prompt against labeled data."*
>
> *— `evals/trajectory_evaluators.py`: "Both flavors here — `trajectory_superset` heuristic and `trajectory_judge` LLM-version using `agentevals`. Heuristic gates CI; LLM-version is for cases where strict superset is too rigid."*
>
> *Switch back to notebook for Cell 9.*

In [ ]:
from langsmith import Client
from evals.dataset import upsert_dataset, DATASET_NAME
from evals.target import target
from evals.heuristic_evaluators import (
    classification_correct, refund_safety, escalation_correctness,
)
from evals.llm_judge_evaluators import kb_grounding_judge
from evals.trajectory_evaluators import trajectory_superset

upsert_dataset()
client = Client()
results = client.evaluate(
    target,
    data=DATASET_NAME,
    evaluators=[
        classification_correct,
        refund_safety,
        escalation_correctness,
        kb_grounding_judge,
        trajectory_superset,
    ],
    experiment_prefix="workshop-live",
    max_concurrency=2,
)
print(results)

# >>> NARRATION (~7 min total — the wow moment):
# >>> Step 1: Run cell. Experiment takes ~3 min. While it runs, narrate:
#     "I'm running the workshop-live experiment now — this is the v3-restored
#      prompt against the dataset. Same evaluators, latest version of the
#      agent prompt. About 3 minutes. While it runs, let me explain what's
#      happening: 21 examples run through the agent, five evaluators score
#      each output, results stream to LangSmith. Three heuristics, one
#      LLM-judge, one trajectory check — same shape we just walked through."
# >>> Step 1b: SIDE-TAB FLASH tests/test_agent_quality.py (~20 sec):
#     "And here's the bridge to CI — these same evaluators run inside pytest.
#      @pytest.mark.langsmith decorator, from langsmith import testing as t,
#      and the assertions hard-fail on refund_safety == 1. That's how the CI
#      gate actually works — pytest IS the runner, results stream to LangSmith
#      under the workshop-pytest-local project, and a failed assertion blocks
#      the PR. Same evaluators, two surfaces — client.evaluate for ad-hoc
#      experiments, pytest for CI. The marker config is in pyproject.toml,
#      the workflow YAML is in .github/workflows/evals.yml — both in the repo."
# >>> Step 2: When done, SWITCH TO PRE-LOADED LANGSMITH COMPARISON VIEW
#     (bookmarked URL with v1/v2/v3 selected). Walk through carefully:
#     - Click v1-baseline first: "All green. Original prompt, 100% pass rate."
#     - Click v2-removed-guardrail: "I deleted the line 'Never refund a
#       free-tier customer' from the system prompt. Look at refund_safety —
#       drops on ex-012. The trap fired."
#     - Click into ex-012 trace: "Here's the agent's reasoning — 'customer
#       is on free tier, but they're asking for a refund... I'll process
#       the refund request.' That's the regression. The agent
#       hallucinated permission to refund."
#     - Click v3-restored: "Line restored. Score back to 1.0. Regression
#       caught and fixed, with the exact failure visible in the trace."
# >>> Step 2b: WHILE ON THE COMPARISON VIEW, point at the example column (~25 sec):
#     "And these example IDs — ex-001, ex-012, ex-021 — those are the seed
#      examples from src/seed_data.py. 21 of them, hand-curated, with stable
#      ex_id metadata so LangSmith joins them across experiments. That's why
#      the Compare view works — same example UUIDs, same dataset, three
#      different agent versions scored side-by-side. Without stable IDs, this
#      view doesn't join. The upsert pattern is in evals/dataset.py — ~30
#      lines, keyed on ex_id metadata so re-runs are idempotent: same UUIDs
#      survive every run, no delete-then-recreate. That's a non-obvious
#      LangSmith ops pattern most tutorials skip."
# >>> Step 3: Pause. Let it land. THEN deliver the framing:
#     "This is the eval-driven development loop in action — a regression
#      caught visually, with the exact failure mode visible in the trace,
#      before any of this hits production code. THIS is what makes the
#      discipline worth the cost."
# >>> Step 3b — PLANT THE CONTRARIAN SEED (~15 sec, sets up the Slide 8 close):
#     "One thing to notice as we move on — what you just saw was a single
#      run. Pass rate from a single run is a snapshot. We'll come back to
#      this in the close — there's a reason that matters."
#     [This sets up Slide 8's contrarian claim: pass rate from a single run
#     hides non-determinism; you need multi-trial to see cross-run stability.]

In [ ]:
# Cell 10: The CI gate — runs the EXACT command GitHub Actions runs on every PR.
#
# Per .github/workflows/evals.yml: `pytest tests/ --langsmith-output -v`
# (no marker filter — both regression AND capability suites run together).
#
# Gating is via ASSERTIONS, not via marker filters. The regression test
# (test_agent_quality) hard-asserts on `refund_safety == 1` — a regression
# fails the assertion and blocks the PR. The capability test has no asserts —
# it runs in the same invocation, logs scores to LangSmith, but never blocks
# merging.
#
# Output below is what CI sees. If it's green here, it'll be green on push.
import subprocess

result = subprocess.run(
    ["uv", "run", "pytest", "tests/", "-v", "--tb=short"],
    capture_output=True, text=True, timeout=300,
)
print(result.stdout[-3500:])  # tail — assertion summary lives here
print(f"\nEXIT CODE: {result.returncode}  (0 = green; nonzero = regression caught)")

## A note on what we just measured

What we ran in cell 9 is a **regression suite** — 21 examples where we know the correct outcome and we expect the agent to pass all of them. That's the production gate against backsliding. Cell 10 just ran pytest as CI does — both suites at once; the green output is what GitHub Actions saw on the PR.

But eval-driven development has a second half: **capability evals** — harder examples where the agent currently struggles. We track pass rate on these *climbing over time* as we improve the agent.

Per LangChain's [Agent Evaluation Readiness Checklist](https://www.langchain.com/blog/agent-evaluation-readiness-checklist):

> Capability evals answer "what can it do?" — start with low pass rate.
>
> Regression evals answer "does it still work?" — should have ~100% pass rate.
>
> You need both because they serve different purposes.

Three deliberately hard examples in `src/seed_data.py`:

- **cap-001**: multi-issue handling
- **cap-002**: context-aware escalation
- **cap-003**: policy synthesis

The next cell uses `pytest -m capability` to run **only** the capability tests in isolation. This is a local-development convenience — useful when you want to iterate on the capability suite without re-running regression — not a separate CI gate. Capability tests have no hard-asserts, so they log scores to LangSmith for tracking but never block a PR. CI runs both suites together (which we already saw in Cell 10); the gate is enforced by the assertions inside regression tests, not by which tests are included.

> *LIVE NARRATION (~75 sec): "Same pytest infrastructure, different gating. Both halves matter — regression catches backsliding, capability tracks improvement. CI runs both on every PR; only regression assertions can fail the gate. The marker filter you'll see next is a local-iteration convenience, not a CI mechanism."*

In [ ]:
# Cell 12: Capability-only run — local iteration convenience, NOT a CI gate.
#
# Cell 10 already ran the capability tests as part of the full CI invocation
# (`pytest tests/`). This cell uses the `-m capability` marker filter to run
# capability ONLY — useful when you're iterating on capability examples and
# don't want to re-run regression every time. The marker filter is a local
# development tool, not a CI configuration choice.
#
# Capability tests have NO hard-asserts: they log scores to LangSmith for
# tracking but never fail the PR. Pass count is informational; the meaningful
# output is the LangSmith experiment that surfaces the score over time as the
# agent improves.
import subprocess

result = subprocess.run(
    ["uv", "run", "pytest", "tests/", "-m", "capability", "-v", "--tb=short"],
    capture_output=True, text=True, timeout=180,
)
print(result.stdout[-3000:])
print(f"\nEXIT CODE: {result.returncode}  (capability never fails CI; pass count is informational)")

## How do we know the LLM judge is right?

We don't, yet. LLM-as-judge scores require **calibration** — comparing judge decisions against human judgments on labeled examples.

LangSmith's workflow for this is the **Align Eval** feature:

1. Open an experiment, find a trace where the judge scored something
2. Click **Add Feedback** — register your human judgment
3. Aggregate ~20+ labeled examples in an Annotation Queue
4. Use **Align Eval** to create or refine the judge prompt FROM THE LABELED DATA

This is why we DON'T hard-assert on judge scores in CI. Heuristic scores (deterministic) gate CI; judge scores get tracked for visibility and recalibrated over time.

> *LIVE WALKTHROUGH (~4 min — DO THIS LIVE):*
>
> 1. SWITCH TO PRE-LOADED LANGSMITH TAB (experiment with low kb_grounding scores)
> 2. Click into a trace, show the judge's score
> 3. Click "Add Feedback" — show the human-judgment workflow
> 4. Mention Annotation Queues by name
> 5. Mention Align Eval explicitly: *"Recalibrate from labeled data, not from prompt-engineering alone."*
> 6. Switch back to notebook. Close on: *"Heuristics gate; judges track."*